## Парсинг данных с сайта на Python

### Определения

Парсинг (англ. parsing — разбор) — это процесс автоматического анализа веб-сайтов для сбора структурированной информации. Еще парсинг часто называют веб-скрапингом. 

Парсинг - это процесс сбора данных с последующей их обработкой и анализом.

Программа, которая занимается парсингом, называют - парсер.


### Условие задачи

С сайта ( https://habr.com/ru/search/ ) необходимо построить исходный набор данных (.csv или .xml). Набор данных должен включать __название, описание, рейтинг и сферу деятельности компаний, дату публикации, а также текст статей из Интернет-ресурсов__. Подготовленный набор данных должен содержать сведения о всех номинантах конкурса. Разработанный парсер должен извлекать гиперссылки из начальной страницы с последующим обходом всех страниц по полученным ссылкам и извлечением их содержимого. Можно дополнить набор какими-либо другими данными, если они могут быть полезны для дальнейшего исследования.


### Этапы парсинга

1. Поиск данных
2. Получение информации
3. Сохранение данных

### Подключение библиотек

In [1]:
from bs4 import BeautifulSoup as bs

Beautiful Soup - это библиотека Python для извлечения данных из HTML и XML файлов. 

In [2]:
import requests

Библиотека requests является стандартным инструментом для составления HTTP-запросов в Python.

In [3]:
import pandas as pd

### Получение информаций

In [4]:
# GET - запрос
url = 'https://habr.com/ru/all/' # страница со всеми статьями 
page = requests.get(url)

Метод __requests.get(url)__ из библиотеки requests в Python выполняет HTTP-запрос типа GET по указанному URL. Этот запрос используется для получения данных с веб-страницы или API, в нашем случае из страницы habr.

In [5]:
page.status_code

200

Если вызвать __page.status_code__, то получим статус состояния HTTP. например, 200 — успешно, 404 — страница не найдена, 500 — ошибка сервера 


In [6]:
soup = bs(page.text, 'html.parser')

__bs__ — это сокращение от BeautifulSoup, основного класса библиотеки Beautiful Soup.

__bs(page.text, 'html.parser')__ создаёт объект BeautifulSoup, который парсит HTML-код из page.text с использованием указанного парсера.

__'html.parser'__ — это встроенный парсер Python, который не требует установки дополнительных библиотек.а.

In [7]:
page.text

'<!DOCTYPE html>\n<html lang="ru">\n\n  <head>\n    <title>Все статьи подряд &#x2F; Хабр</title>\n<link rel="image_src" href="/img/habr_ru.png" data-hid="2a79c45">\n<link href="https://habr.com/ru/articles/" rel="canonical" data-hid="e3fa780">\n<link href="https://habr.com/ru/articles/" hreflang="ru" rel="alternate" data-hid="7d51b8a">\n<link href="https://habr.com/en/articles/" hreflang="en" rel="alternate" data-hid="7d51b8a">\n<meta itemprop="image" content="/img/habr_ru.png">\n<meta property="og:image" content="/img/habr_ru.png">\n<meta property="og:image:width" content="1200">\n<meta property="og:image:height" content="630">\n<meta property="aiturec:image" content="/img/habr_ru.png">\n<meta name="twitter:image" content="/img/habr_ru.png">\n<meta property="vk:image" content="/img/habr_ru.png?format=vk">\n<meta property="fb:app_id" content="444736788986613">\n<meta property="fb:pages" content="472597926099084">\n<meta name="twitter:card" content="summary_large_image">\n<meta name="tw

In [8]:
soup

<!DOCTYPE html>

<html lang="ru">
<head>
<title>Все статьи подряд / Хабр</title>
<link data-hid="2a79c45" href="/img/habr_ru.png" rel="image_src"/>
<link data-hid="e3fa780" href="https://habr.com/ru/articles/" rel="canonical"/>
<link data-hid="7d51b8a" href="https://habr.com/ru/articles/" hreflang="ru" rel="alternate"/>
<link data-hid="7d51b8a" href="https://habr.com/en/articles/" hreflang="en" rel="alternate"/>
<meta content="/img/habr_ru.png" itemprop="image"/>
<meta content="/img/habr_ru.png" property="og:image"/>
<meta content="1200" property="og:image:width"/>
<meta content="630" property="og:image:height"/>
<meta content="/img/habr_ru.png" property="aiturec:image"/>
<meta content="/img/habr_ru.png" name="twitter:image"/>
<meta content="/img/habr_ru.png?format=vk" property="vk:image"/>
<meta content="444736788986613" property="fb:app_id"/>
<meta content="472597926099084" property="fb:pages"/>
<meta content="summary_large_image" name="twitter:card"/>
<meta content="@habr_com" name=

Создадим словарь, в который будем записывать данные по заданию: название статьи, описание, рейтинг и сферу деятельности компаний, дату публикации, а также текст статьи из Интернет-ресурса

In [9]:
result_list = {'title': [], 'namecompany': [], 'description': [], 'rating': [], 'field': [], 'date': [], 'textpub': []}

### Алгоритм

Суть алгоритма заключается в переборе страниц, и переходе на "вложенные" страницы, то есть у нас есть основная страница https://habr.com/ru/all/, мы перебираем несколько стараниц с page1 до page10. На каждой странице есть статьи, записываем их в список, чтобы перейти по ним используем  _-i.a.get('href')-_  то есть берём значение из href этого заголовка. Далее находим классы элементов которые нам нужны, и записываем их в результат.

In [10]:
pagenum = 1
for i in range(10):
    url = 'https://habr.com/ru/feed/page' + str(pagenum) + '/' # переход на ссылуку с определённым номером сраницы
    page = requests.get(url)
    soup = bs(page.text, 'html.parser')
    titles = soup.find_all('h2', class_='tm-title tm-title_h2')# получаем заголовки всех статей на этой странице
    
    for i in titles: 
        # переход на страницу статьи
        url = 'https://habr.com' + str(i.a.get('href')) 
        page = requests.get(url)
        soup = bs(page.text, 'html.parser')
        
        name_company = soup.find('a', class_='tm-company-snippet__title')# получаем название компаний
        desc_company = soup.find('div', class_='tm-company-snippet__description')# получаем описание компаний
        
        if (name_company is not None): #если на странице присутсвует компания
        
            result_list['title'].append(i.text) # записываем название с\татьи
            result_list['namecompany'].append(name_company.text) # записываем название компании
            result_list['description'].append(desc_company.text) # записываем описание компании
            
            datepub = soup.find('span', class_='tm-article-datetime-published') # находим дату публикаций
            result_list['date'].append(datepub.time['datetime'][0: 10]) # записываем дату публикаций
            
            # текст статьи
            try:
                textpub = soup.find('div', class_='article-formatted-body article-formatted-body article-formatted-body_version-2').get_text()
                textpub = textpub.replace('\n', ' ').replace('\t', ' ').replace('\xa0', ' ').replace('\u200e', ' ').replace('\r', ' ')
            except:
                textpub = soup.find('div', class_='article-formatted-body article-formatted-body article-formatted-body_version-1').get_text()
                textpub = textpub.replace('\n', ' ').replace('\t', ' ').replace('\xa0', ' ').replace('\u200e', ' ').replace('\r', ' ')
            result_list['textpub'].append(textpub)
            
            # переход на страницу компании
            url = 'https://habr.com' + str(name_company.get('href'))
            page = requests.get(url)
            soup = bs(page.text, 'html.parser')
            
            #записываем рейтинг
            rating = soup.find('span', class_='tm-votes-lever__score-counter tm-votes-lever__score-counter_rating tm-votes-lever__score-counter')
            if(rating is None):
                result_list['rating'].append('0')
            else:
                result_list['rating'].append((rating.text).strip())
               
             #записываем отрасли компаний
            fieldtext = ""
            fields = soup.find_all('a', 'tm-company-profile__categories-text')
            for field in fields:
                fieldtext = fieldtext + ((field.text).strip()) + ", "
            if (fields is None):
                result_list['field'].append(None)
            else:
                result_list['field'].append(fieldtext[0:-2])
            
    pagenum += 1

Что бы найти элемет на странице, выделите этот элемент(заголовок, текст, изображение) и нажмите Ctrl + Shift + I или ПКМ и "Исследовать элемент".

In [11]:
result_list

{'title': ['Владислав Бакальчук стал генеральным директором «М.Видео»',
  'RxJS в Angular: 5 операторов, которые превращают хаос данных в симфонию',
  'У американских законодателей возникли вопросы к OpenAI из-за сделки с Пентагоном',
  'Где брать операторов поддержки, которых не заменят чат-боты',
  'Фамипия. Оживляем раритетный домофон с тремя ручками',
  'Лечение переломов: 3D-печать гидрогелевого импланта',
  'Линейка HighFreq или как выжать из облака максимум для инференса, ML и других высоких нагрузок',
  'Практика защиты ЦОД от DDoS-атак: схемы интеграции локального комплекса',
  'Можно ли будет благодаря ИИ обойтись без менеджеров пакетов?',
  'Темная эра онлайна: как сохранить в ней маркетинг и не поддаться негативным трендам',
  'Вершина пирамиды безопасности: обзор российских NAC-решений для контроля сетевого доступа',
  'Как качать жидкости без насоса: эрлифт/гейзерный насос',
  'АТОМ ID получил сертификат ФСТЭК России 4-го уровня доверия',
  'Чтение на выходные: «Взломавша

In [12]:
print("Количество нулевых значений в: ")
for i in result_list:
    print( i + " - " + str(result_list[i].count(None)))

Количество нулевых значений в: 
title - 0
namecompany - 0
description - 0
rating - 0
field - 0
date - 0
textpub - 0


### Сохранение данных

In [13]:
file_name = 'habr.csv'
df = pd.DataFrame(data=result_list)
df.to_csv(file_name)

In [14]:
df.head(15)

,title,namecompany,description,rating,field,date,textpub
0,Владислав Бакальчук стал генеральным директоро...,М.Видео-Эльдорадо,30 лет в топе,103.68,Электронная коммерция,2026-03-13,Совет директоров «М.Видео» избрал Феликса Либа...
1,"RxJS в Angular: 5 операторов, которые превраща...",RUVDS.com,VDS/VPS-хостинг. Скидка 15% по коду HABR15,3055.47,"Связь и телекоммуникации, Домены и хостинг, Ве...",2026-03-13,Стоит начать с боли всех разработчиков Angular...
2,У американских законодателей возникли вопросы ...,BotHub,Российский Openrouter,429.67,"Веб-разработка, Программное обеспечение, Веб-с...",2026-03-13,Спешно заключенная OpenAI сделка с Министерств...
3,"Где брать операторов поддержки, которых не зам...",TEAMLY,Компания,24.17,"Программное обеспечение, Веб-сервисы",2026-03-13,"Чат-боты победили рутину, но вместе с ней унич..."
4,Фамипия. Оживляем раритетный домофон с тремя р...,Timeweb Cloud,То самое облако,2178.95,Связь и телекоммуникации,2026-03-13,Приветствую всех!Не так давно я уже рассказыва...
5,Лечение переломов: 3D-печать гидрогелевого имп...,ua-hosting.company,Хостинг-провайдер: серверы в NL до 300 Гбит/с,91.44,"Аппаратное обеспечение, Связь и телекоммуникац...",2026-03-13,Перелом кости является одной из самых распрост...
6,Линейка HighFreq или как выжать из облака макс...,Selectel,IT-инфраструктура для бизнеса,2022.86,"Аппаратное обеспечение, Связь и телекоммуникац...",2026-03-13,«Больше» — не всегда значит «лучше». К пользов...
7,Практика защиты ЦОД от DDoS-атак: схемы интегр...,Компания «Гарда»,Защита данных и сетевая безопасность,68.63,Информационная безопасность,2026-03-13,За последние пять лет российский рынок дата-це...
8,Можно ли будет благодаря ИИ обойтись без менед...,Издательский дом «Питер»,Компания,175.65,"СМИ, Электронная коммерция, Производство мульт...",2026-03-13,"Недавно Марсело Эммерих написал пост, в которо..."
9,Темная эра онлайна: как сохранить в ней маркет...,МТС,Про жизнь и развитие в IT,1710.49,"Связь и телекоммуникации, Мобильные технологии...",2026-03-13,"Привет, Хабр! Меня зовут Андрей Аврамчук, я ре..."


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        59 non-null     object
 1   namecompany  59 non-null     object
 2   description  59 non-null     object
 3   rating       59 non-null     object
 4   field        59 non-null     object
 5   date         59 non-null     object
 6   textpub      59 non-null     object
dtypes: object(7)
memory usage: 3.4+ KB


# Работа с IMDb

In [9]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup as bs
import pandas as pd
import re, time

In [13]:
driver = uc.Chrome(options=uc.ChromeOptions(), version_main=146)
driver.set_page_load_timeout(60)
driver.get('https://www.imdb.com/chart/top')
time.sleep(5)

In [14]:
soup = bs(driver.page_source, 'html.parser') 
tt_ids = list(dict.fromkeys(re.findall(r'/(tt\d+)/', str(soup))))[:250]
film_links = [f'https://www.imdb.com/title/{tt}/' for tt in tt_ids]
print(f'Найдено фильмов: {len(film_links)}')

Найдено фильмов: 250


In [15]:
EXCLUDE = {'Browse movies by genre', 'Browse TV shows by genre', 'Related interests', ''}

In [16]:
def parse_film(url):
    driver.get(url)
    time.sleep(1.5) 
    soup = bs(driver.page_source, 'html.parser')

    def get(tag, attr):
        el = soup.find(tag, attr)
        return el.get_text(strip=True) if el else ''

    title  = get('span', {'data-testid': 'hero__primary-text'}) or get('h1', {})
    
    rating = get('div', {'data-testid': 'hero-rating-bar__aggregate-rating__score'})
    rating = rating.replace('/10', '')

    genres_raw = [a.get_text(strip=True) for a in soup.find_all('a', href=re.compile(r'/interest/'))]
    seen = set()
    genres = []
    for g in genres_raw:
        if g not in EXCLUDE and g not in seen:
            seen.add(g)
            genres.append(g)
    genre = ', '.join(genres)

    desc = get('span', {'data-testid': 'plot-xs_to_m'}) or get('p', {'data-testid': 'plot'})

    year_tag = soup.find('a', href=re.compile(r'/releaseinfo'))
    year = re.search(r'\d{4}', year_tag.get_text() if year_tag else '')
    year = year.group() if year else ''

    return {'title': title, 'year': int(year) if year else None,
            'rating': rating, 'genre': genre, 'description': desc} if title else None

results = []
for i, link in enumerate(film_links, 1):
    film = parse_film(link)
    if film:
        results.append(film)
        print(film)

driver.quit()

df = pd.DataFrame(results)
print(df.head(10))

{'title': 'Побег из Шоушенка', 'year': 1994, 'rating': '9.3', 'genre': 'Period Drama, Prison Drama, Drama', 'description': 'A wrongfully convicted banker forms a close friendship with a hardened convict over a quarter century while retaining his humanity through simple acts of compassion.'}
{'title': 'Крестный отец', 'year': 1972, 'rating': '9.2', 'genre': 'Epic, Gangster, Psychological Drama, Tragedy, Crime, Drama', 'description': 'The aging patriarch of an organized crime dynasty transfers control of his clandestine empire to his reluctant son.'}
{'title': 'Тёмный рыцарь', 'year': 2008, 'rating': '9.1', 'genre': 'Action Epic, Epic, Psychological Drama, Psychological Thriller, Superhero, Tragedy, Action, Crime, Drama, Thriller', 'description': 'When a menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman, James Gordon and Harvey Dent must work together to put an end to the madness.'}
{'title': 'Крестный отец 2', 'year': 1974, 'rating': '9.0', 'genre': 'Epic,

In [17]:
df.head(15)

,title,year,rating,genre,description
0,Побег из Шоушенка,1994,9.3,"Period Drama, Prison Drama, Drama",A wrongfully convicted banker forms a close fr...
1,Крестный отец,1972,9.2,"Epic, Gangster, Psychological Drama, Tragedy, ...",The aging patriarch of an organized crime dyna...
2,Тёмный рыцарь,2008,9.1,"Action Epic, Epic, Psychological Drama, Psycho...",When a menace known as the Joker wreaks havoc ...
3,Крестный отец 2,1974,9.0,"Epic, Gangster, Tragedy, Crime, Drama",The early life and career of Vito Corleone in ...
4,12 разгневанных мужчин,1957,9.0,"Legal Drama, Psychological Drama, Crime, Drama",The jury in a New York City murder trial is fr...
5,Властелин колец: Возвращение короля,2003,9.0,"Action Epic, Adventure Epic, Epic, Fantasy Epi...",Gandalf and Aragorn lead the World of Men agai...
6,Список Шиндлера,1993,9.0,"Docudrama, Epic, Historical Epic, Period Drama...","In German-occupied Poland during World War II,..."
7,Властелин колец: Братство кольца,2001,8.9,"Action Epic, Adventure Epic, Dark Fantasy, Epi...",A meek Hobbit from the Shire and eight compani...
8,Криминальное чтиво,1994,8.8,"Dark Comedy, Drug Crime, Gangster, Crime, Drama","The lives of two mob hitmen, a boxer, a gangst..."
9,"Хороший, плохой, злой",1966,8.8,"Action Epic, Adventure Epic, Dark Comedy, Dese...",A bounty-hunting scam joins two men in an unea...


In [18]:
print("Количество нулевых значений в:")
for col in df.columns:
    print(f"  {col} - {df[col].isna().sum()}")

Количество нулевых значений в:
  title - 0
  year - 0
  rating - 0
  genre - 0
  description - 0


In [21]:
df.to_csv("top250filmsInIMDb.csv", index=False, encoding='utf-8-sig')

# API

In [45]:
import requests
import pandas as pd

API_KEY = "4KQYX9V-M2FMWH5-Q7GT3FW-Y2PCQKK"
API_HEADERS = {'X-API-KEY': API_KEY}

In [48]:
r = requests.get(
    'https://api.poiskkino.dev/v1.4/movie',
    headers=API_HEADERS,
    params=[
        ('lists', 'top250'),
        ('sortField[]', 'top250'),
        ('sortType[]', '1'),
        ('selectFields[]', 'name'),
        ('selectFields[]', 'year'),
        ('selectFields[]', 'rating'),
        ('selectFields[]', 'genres'),
        ('selectFields[]', 'description'),
        ('selectFields[]', 'top250'),
        ('limit', 250),
        ('page', 1),
    ],
    timeout=15
)

films = r.json().get('docs', [])
for f in films[:5]:
    print(f'{f.get("top250")}. {f.get("name")}')

None. Зеленая книга
None. Остров проклятых
None. Интерстеллар
None. Освобождение: Огненная дуга
None. На войне как на войне


In [51]:
result_list = {
    'title': [], 'genre': [], 'year': [],
    'rating': [], 'description': []
}

for film in films:
    title       = film.get('name', '')
    year        = film.get('year')
    rating      = round(film.get('rating', {}).get('kp') or 0, 1)
    genre       = ', '.join(g['name'] for g in film.get('genres', []) if g.get('name'))
    description = (film.get('description') or '').strip()

    if not title:
        continue

    result_list['title'].append(title)
    result_list['genre'].append(genre)
    result_list['year'].append(int(year) if year else None)
    result_list['rating'].append(rating)
    result_list['description'].append(description)

df_kp = pd.DataFrame(result_list).sort_values('rating', ascending=False).reset_index(drop=True)
print(df_kp.shape)
df_kp.head(10)

(250, 5)


,title,genre,year,rating,description
0,Зеленая миля,"драма, фэнтези, криминал",1999,9.1,Пол Эджкомб — начальник блока смертников в тюр...
1,Побег из Шоушенка,драма,1994,9.1,Бухгалтер Энди Дюфрейн обвинён в убийстве собс...
2,Форрест Гамп,"драма, комедия, мелодрама, история, военный",1994,8.9,"Сидя на автобусной остановке, Форрест Гамп — н..."
3,1+1,"драма, комедия",2011,8.9,"Пострадав в результате несчастного случая, бог..."
4,Список Шиндлера,"биография, история, драма, военный",1993,8.9,"Оскар Шиндлер, член национал-социалистической ..."
5,Операция «Ы» и другие приключения Шурика,"комедия, мелодрама, криминал",1965,8.8,Студент Шурик попадает в самые невероятные сит...
6,Бойцовский клуб,"триллер, драма, криминал",1999,8.7,Сотрудник страховой компании страдает хроничес...
7,Джентльмены,"криминал, комедия, боевик",2019,8.7,Один ушлый американец ещё со студенческих лет ...
8,Интерстеллар,"фантастика, драма, приключения",2014,8.7,"Когда засуха, пыльные бури и вымирание растени..."
9,Леон,"боевик, триллер, драма, криминал",1994,8.7,Профессиональный убийца Леон неожиданно для се...


In [52]:
df_kp.to_csv('kinopoisk_top250.csv', index=False, encoding='utf-8-sig')

In [53]:
df_combined = pd.concat([df, df_kp], ignore_index=True).drop_duplicates(subset='title').reset_index(drop=True)
print(df_combined.shape)
df_combined.head(10)

(419, 5)


,title,year,rating,genre,description
0,Побег из Шоушенка,1994,9.3,"Period Drama, Prison Drama, Drama",A wrongfully convicted banker forms a close fr...
1,Крестный отец,1972,9.2,"Epic, Gangster, Psychological Drama, Tragedy, ...",The aging patriarch of an organized crime dyna...
2,Тёмный рыцарь,2008,9.1,"Action Epic, Epic, Psychological Drama, Psycho...",When a menace known as the Joker wreaks havoc ...
3,Крестный отец 2,1974,9.0,"Epic, Gangster, Tragedy, Crime, Drama",The early life and career of Vito Corleone in ...
4,12 разгневанных мужчин,1957,9.0,"Legal Drama, Psychological Drama, Crime, Drama",The jury in a New York City murder trial is fr...
5,Властелин колец: Возвращение короля,2003,9.0,"Action Epic, Adventure Epic, Epic, Fantasy Epi...",Gandalf and Aragorn lead the World of Men agai...
6,Список Шиндлера,1993,9.0,"Docudrama, Epic, Historical Epic, Period Drama...","In German-occupied Poland during World War II,..."
7,Властелин колец: Братство кольца,2001,8.9,"Action Epic, Adventure Epic, Dark Fantasy, Epi...",A meek Hobbit from the Shire and eight compani...
8,Криминальное чтиво,1994,8.8,"Dark Comedy, Drug Crime, Gangster, Crime, Drama","The lives of two mob hitmen, a boxer, a gangst..."
9,"Хороший, плохой, злой",1966,8.8,"Action Epic, Adventure Epic, Dark Comedy, Dese...",A bounty-hunting scam joins two men in an unea...


In [54]:
df_combined.to_csv('combined_top250.csv', index=False, encoding='utf-8-sig')